# Differential Equations — Session 2  
## Section 1.2: Initial-Value Problems, Existence, and Uniqueness

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture notebook

### Learning objectives

Students should be able to:

1. State first- and higher-order initial-value problems.
2. Interpret initial conditions geometrically and physically.
3. Select a particular solution from a family.
4. Explain the difference between existence and uniqueness.
5. Apply the standard first-order existence–uniqueness test.
6. Recognize nonuniqueness and finite-time blow-up.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Activity |
|---:|---|
| 0–12 min | Meaning of initial data |
| 12–30 min | First-order IVPs and solution-family visualization |
| 30–45 min | Second-order IVPs |
| 45–63 min | Existence and uniqueness theorem |
| 63–77 min | Nonunique solutions |
| 77–87 min | Maximal interval and blow-up |
| 87–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display
from scipy.integrate import solve_ivp

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=4, suppress=True)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## 1. Initial-value problems

A first-order IVP has the form

$$
y'=f(x,y),\qquad y(x_0)=y_0.
$$

Geometrically, we seek a solution curve that passes through $(x_0,y_0)$.

A second-order IVP has two conditions:

$$
y''=f(x,y,y'),\qquad
y(x_0)=y_0,\qquad
y'(x_0)=y_1.
$$

For mechanical motion, these usually specify initial position and velocity.

### Classroom Checkpoint — Existence Versus Uniqueness

Suppose $f(x,y)$ is continuous near $(x_0,y_0)$, but no Lipschitz condition in $y$ is known. What conclusion is justified for the IVP $y'=f(x,y)$?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 2. Selecting one member of a first-order family

The family

$$
y=Ce^{x/2}
$$

solves

$$
y'=\frac12y.
$$

Impose $y(1)=4$:

$$
4=Ce^{1/2}
\quad\Longrightarrow\quad
C=4e^{-1/2}.
$$

In [ ]:
def first_order_ivp_picture(x0=1.0, y0=4.0):
    x_vals = np.linspace(-2, 4, 500)
    for C in [-4, -2, 0, 2, 4]:
        plt.plot(x_vals, C*np.exp(x_vals/2), alpha=0.55)
    C_selected = y0*np.exp(-x0/2)
    plt.plot(x_vals, C_selected*np.exp(x_vals/2),
             linestyle="--", linewidth=3,
             label=fr"selected: $C={C_selected:.3f}$")
    plt.scatter([x0], [y0], s=80, label=fr"initial point $({x0:.1f},{y0:.1f})$")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(r"An initial condition selects one curve from $y=Ce^{x/2}$")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        first_order_ivp_picture,
        x0=FloatSlider(min=-1, max=2, step=0.25, value=1),
        y0=FloatSlider(min=-5, max=5, step=0.5, value=4)
    )
else:
    first_order_ivp_picture()

## 3. Second-order IVP

The family

$$
x(t)=c_1\cos(2t)+c_2\sin(2t)
$$

solves $x''+4x=0$.

Suppose

$$
x(0)=1.5,\qquad x'(0)=-2.
$$

Then $c_1=1.5$, and because

$$
x'(t)=-2c_1\sin(2t)+2c_2\cos(2t),
$$

we obtain $2c_2=-2$, so $c_2=-1$.

In [ ]:
t = np.linspace(0, 3*np.pi, 800)
position = 1.5*np.cos(2*t) - np.sin(2*t)
velocity = -3*np.sin(2*t) - 2*np.cos(2*t)

plt.plot(t, position, label="position x(t)")
plt.plot(t, velocity, linestyle="--", label="velocity x'(t)")
plt.scatter([0], [1.5], s=70)
plt.xlabel("t")
plt.ylabel("state")
plt.title("Second-order IVP: initial position and velocity")
plt.legend()
plt.show()

## 4. Existence and uniqueness

For

$$
y'=f(x,y),\qquad y(x_0)=y_0,
$$

a standard local theorem says:

If both $f$ and $f_y=\partial f/\partial y$ are continuous in a rectangle containing $(x_0,y_0)$, then the IVP has a **unique local solution**.

### Interpretation

- Continuity of $f$ supports **existence**.
- Regular dependence on $y$, commonly checked by continuity of $f_y$, supports **uniqueness**.
- The theorem is sufficient, not necessary.
- It guarantees a solution on some interval around $x_0$, not necessarily for all $x$.

In [ ]:
# Geometry of a theorem rectangle for f(x,y)=x-y
x_grid = np.linspace(-2, 2, 17)
y_grid = np.linspace(-1.5, 2.5, 17)
X, Y = np.meshgrid(x_grid, y_grid)
S = X - Y
U = np.ones_like(S)
L = np.sqrt(U**2 + S**2)
U, V = U/L, S/L

plt.quiver(X, Y, U, V, angles="xy")
plt.plot([-1.5, 1.5, 1.5, -1.5, -1.5],
         [-1, -1, 2, 2, -1], linewidth=2, label="continuity rectangle")
plt.scatter([0], [0.5], s=80, label="initial point")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Direction field inside a rectangle: $y'=x-y$")
plt.legend()
plt.show()

## 5. What can go wrong? Nonuniqueness

Consider

$$
y'=3y^{2/3},\qquad y(0)=0.
$$

Both

$$
y_1(x)=0
\qquad\text{and}\qquad
y_2(x)=x^3
$$

satisfy the IVP.

More generally, for any $a\ge0$,

$$
y_a(x)=
\begin{cases}
0, & x\le a,\\
(x-a)^3, & x>a
\end{cases}
$$

is a solution passing through $(0,0)$. The solution can remain at equilibrium for an arbitrary waiting time and then depart.

In [ ]:
def delayed_solution(x, a):
    return np.where(x <= a, 0.0, (x-a)**3)

x_vals = np.linspace(-1, 3, 600)
for a in [0, 0.5, 1.0, 1.5, 2.0]:
    plt.plot(x_vals, delayed_solution(x_vals, a), label=f"wait until a={a:g}")

plt.plot(x_vals, np.zeros_like(x_vals), linestyle="--", linewidth=2,
         label="remain at y=0")
plt.scatter([0], [0], s=80)
plt.xlabel("x")
plt.ylabel("y")
plt.title("One IVP with infinitely many solutions")
plt.legend()
plt.show()

Why does the theorem fail?

$$
f(y)=3y^{2/3}
$$

is continuous, but

$$
f_y(y)=2y^{-1/3}
$$

is not continuous at $y=0$. Thus the standard uniqueness condition is absent at the initial point.

## 6. Local existence does not mean global existence

Consider

$$
y'=y^2,\qquad y(0)=y_0.
$$

For $y_0\ne0$,

$$
y(x)=\frac{y_0}{1-y_0x}.
$$

When $y_0>0$, the solution blows up at $x=1/y_0$. The theorem guarantees local existence and uniqueness, but the maximal interval stops at the vertical asymptote.

In [ ]:
def blowup_plot(y0=1.0):
    if abs(y0) < 1e-10:
        x_vals = np.linspace(-4, 4, 500)
        plt.plot(x_vals, np.zeros_like(x_vals), label="y=0")
        interval_text = "all real x"
    else:
        singular = 1/y0
        left = np.linspace(max(-5, singular-6), singular-0.04, 500)
        right = np.linspace(singular+0.04, min(5, singular+6), 500)
        if len(left) > 1:
            plt.plot(left, y0/(1-y0*left))
        if len(right) > 1:
            plt.plot(right, y0/(1-y0*right))
        plt.axvline(singular, linestyle="--", label=f"blow-up x={singular:.2f}")
        interval_text = (
            f"maximal interval containing x=0: (-∞,{singular:.2f})"
            if singular > 0 else
            f"maximal interval containing x=0: ({singular:.2f},∞)"
        )
    plt.scatter([0], [y0], s=70, label=f"initial value y(0)={y0:g}")
    plt.ylim(-10, 10)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(r"$y'=y^2$: unique locally, possibly not global")
    plt.legend()
    plt.show()
    print(interval_text)

if WIDGETS_AVAILABLE:
    interact(
        blowup_plot,
        y0=FloatSlider(min=-2, max=2, step=0.25, value=1)
    )
else:
    blowup_plot(1.0)

## 7. Numerical check

A numerical solver can approximate the solution while it remains finite. It cannot continue reliably through a true blow-up.

In [ ]:
def rhs(x, y):
    return y**2

sol = solve_ivp(rhs, (0, 0.9), [1.0], dense_output=True, max_step=0.03)
x_eval = np.linspace(0, 0.9, 300)

plt.plot(x_eval, sol.sol(x_eval)[0], label="numerical")
plt.plot(x_eval, 1/(1-x_eval), linestyle="--", label="exact")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Numerical and exact solutions of $y'=y^2,\ y(0)=1$")
plt.legend()
plt.show()

## Exit check

For

$$
y'=\frac{x+y}{x-2},\qquad y(0)=1,
$$

answer:

1. Are $f$ and $f_y$ continuous near $(0,1)$?
2. Does the standard theorem give a unique local solution?
3. Can the theorem guarantee that the solution crosses $x=2$?

**Discussion:** Yes; yes; no. The differential equation itself is undefined at $x=2$.

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.